# BİL458 Computer Vision · YOLO26n Traffic Sign Detection and Edge Deployment

640 px ana deney, 320 px çözünürlük ablasyonudur. Notebook, kaydedilmiş deney çıktılarından tablo ve grafik üretir. Veri setini önce README'deki komutla hazırlayın. Çalıştırma modeli eğitmez ve inference benchmark başlatmaz.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'src/summarize_benchmark.py').is_file())
FINAL = ROOT / 'results/final'
FAILURE = ROOT / 'results/failure_analysis'

def load_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

## GTSDB veri seti

Resmî test aralığı korunur. Prohibitory, danger ve mandatory YOLO sınıflarıdır. Grafikler ve tablolar hazırlanmış veri setinin raporunu kullanır.

In [ ]:
dataset = load_json(ROOT / 'data/gtsdb/preparation_report.json')['split']
splits = ['train', 'val', 'test']
dataset_table = pd.DataFrame(dataset['class_distribution']).T.loc[splits]
dataset_table.insert(0, 'images', pd.Series(dataset['image_counts']))
display(dataset_table)
display(Image(filename=str(ROOT / 'results/dataset_validation.jpg'), width=850))

## Eğitim

Her çözünürlükte seed 0, 1 ve 2 ile ayrı eğitim koşusu vardır. Eğriler validasyon split'inden gelir. Tablodaki en iyi gözlenen validasyon epoch'u mAP50–95 üzerinden gösterilir.

In [ ]:
train_rows = []
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True, layout='constrained')
for resolution, axis in zip([640, 320], axes):
    for seed in range(3):
        history = pd.read_csv(ROOT / f'runs/train/yolo26n_img{resolution}_seed{seed}/results.csv')
        scores = history['metrics/mAP50-95(B)']
        best = history.loc[scores.idxmax()]
        train_rows.append({'resolution_px': resolution, 'seed': seed, 'epochs_run': len(history),
                           'best_val_epoch': int(best['epoch']), 'val_mAP50': best['metrics/mAP50(B)'],
                           'val_mAP50_95': best['metrics/mAP50-95(B)']})
        axis.plot(history['epoch'], scores, label=f'Seed {seed}')
    axis.set(title=f'{resolution} px', xlabel='Epoch', ylabel='Validation mAP50–95')
    axis.legend()
display(pd.DataFrame(train_rows).round(4))
plt.show()

## ONNX export ve INT8 quantization

FP32 export, sabit boyutlu batch-1 one-to-many başlığı kullanır. Static INT8 QDQ kalibrasyonu eğitim görüntülerinden 100 örnek kullanır (kalibrasyon seed 42). Aşağıdaki model dosyası boyutları diskteki ONNX dosyalarından hesaplanır.

In [ ]:
model_rows = []
for resolution in [640, 320]:
    for seed in range(3):
        for variant in ['fp32', 'int8']:
            file = ROOT / f'models/yolo26n_img{resolution}_seed{seed}_{variant}.onnx'
            model_rows.append({'model': file.name, 'MiB': file.stat().st_size / 2**20})
display(pd.DataFrame(model_rows).round(4))

## Raspberry Pi 5 ABBA benchmark

Her seed/model için iki ABBA koşusu ortalanır. Ortalama ve örneklem standart sapması üç eğitim seed'i üzerinden hesaplanır. Her koşuda 227 görüntü × 5 zamanlamalı geçiş vardır. FPS yalnız ONNX inference süresinden türetilir. 640 px ana bulgular aşağıda, 320 px sonuçları ablasyon bölümündedir.

In [ ]:
metrics = pd.read_csv(FINAL / 'summary_metrics.csv')
fields = {'mAP50': 'mAP50', 'mAP50_95': 'mAP50–95', 'inference_mean_ms': 'Inference ms',
          'inference_fps': 'Inference FPS', 'model_mib': 'Model MiB'}

def benchmark_table(resolution):
    rows = metrics.loc[metrics.resolution_px.eq(resolution)].set_index('format')
    return pd.DataFrame({
        label: [f'{mean:.4f} ± {sd:.4f}'
                for mean, sd in zip(rows[f'{field}_mean'], rows[f'{field}_std'])]
        for field, label in fields.items()
    }, index=rows.index)

display(benchmark_table(640))
display(pd.read_csv(FINAL / 'system_health_summary.csv').round(3))

main = metrics[metrics.resolution_px.eq(640)].set_index('format')
fig, axes = plt.subplots(1, 2, figsize=(8, 3), layout='constrained')
for axis, field in zip(axes, ('mAP50_95', 'inference_mean_ms')):
    main[f'{field}_mean'].plot.bar(ax=axis, yerr=main[f'{field}_std'], capsize=4,
                                 rot=0, color=['#3878a3', '#d38038'])
    axis.set(ylabel=fields[field], xlabel='')
plt.show()

## 320 px çözünürlük ablasyonu

Çözünürlük karşılaştırması aynı format için 640 px'i referans alır. 320 px koşusunda ayrı eğitilmiş checkpoint'ler kullanılmıştır. Çubukların birimi referansa göre yüzde değişimdir.

In [ ]:
display(benchmark_table(320))
effects = pd.read_csv(FINAL / 'effects.csv')
resolution_effect = effects.query("effect_type == 'resolution'")
changes = resolution_effect.pivot(index='metric', columns='series', values='percent_change')
display(changes.round(2))
ax = changes.rename(index=fields).plot.barh(figsize=(8, 3.5))
ax.set(xlabel='640 px referansına göre değişim (%)', ylabel='')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## 640 px küçük nesne analizi

Gruplar orijinal görüntüdeki ground-truth kutu alanı tertilleridir. Eşleştirme aynı sınıfta, confidence ≥ 0.25 ve IoU ≥ 0.50 ile yapılır. 320 px için boyuta göre recall sonucu mevcut değildir.

In [ ]:
small = pd.read_csv(FINAL / 'small_object_analysis.csv').set_index('size_group')
display(small[['gt_count', 'fp32_recall_pct', 'int8_recall_pct']].round(2))
means = small[['fp32_recall_mean', 'int8_recall_mean']].rename(
    columns={'fp32_recall_mean': 'FP32', 'int8_recall_mean': 'INT8'})
sd = small[['fp32_recall_std', 'int8_recall_std']].set_axis(means.columns, axis=1)
ax = (100 * means).plot.bar(yerr=100 * sd, capsize=3, rot=0, figsize=(7, 3),
                           color=['#3878a3', '#d38038'])
ax.set(ylabel='Recall (%)', xlabel='GT kutu alanı grubu', ylim=(0, 105))
plt.tight_layout()
plt.show()

## Hata analizi

Yerel CPU'da 227 test görüntüsü üzerinde sabit seed 0 ile üretilen nitel örnekler. Her figür üstte tam sahneyi, altta aynı bölgeyi yakınlaştırarak gösterir. Sınıflandırma ve lokalizasyon olayları, kalan eşleşmemiş FP ve FN olaylarından ayrı sayılır. Bunlar örnek vaka incelemeleridir.

In [ ]:
failure = load_json(FAILURE / 'examples.json')
settings = failure['metadata']
display(pd.Series({key: settings[key] for key in
                   ('seed', 'confidence_strict_gt', 'match_iou', 'nms_iou', 'localization_iou_floor')}))
display(pd.DataFrame(failure['diagnostic_counts']).fillna(0).astype(int))
display(pd.DataFrame(failure['examples'])[['image_id', 'model', 'kind', 'iou']])
for example in failure['examples']:
    display(Image(filename=str(FAILURE / example['figure']), width=1150))